In [1]:
!pip install -U spacy==3.7.2 spacy-lookups-data
!pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 35.3 MB/s eta 0:00:00


In [10]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import ast
import random
import spacy
from spacy.training import Example
from spacy.util import minibatch, compounding


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
def to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x)
            return v if isinstance(v, list) else [x]
        except:
            return [x]
    return []

def steps_to_text(steps_val):
    if isinstance(steps_val, list):
        return " ".join(str(s) for s in steps_val)
    if isinstance(steps_val, str):
        try:
            v = ast.literal_eval(steps_val)
            if isinstance(v, list):
                return " ".join(str(s) for s in v)
        except:
            return steps_val
    return str(steps_val)

def clean_substrings(ings):
    ings = sorted(set(ings), key=len, reverse=True)  # largos primero
    final = []
    for ing in ings:
        if not any(ing != other and ing in other for other in ings):
            final.append(ing)
    return final

In [36]:
df = pd.read_csv("/content/drive/MyDrive/DIET-IA/dataset/DATASET FINAL/RAW_recipes.csv")
print(df.head()[["name","ingredients","steps"]])

# Vocabulario de ingredientes
all_ingredients = []
for ing in df['ingredients'].dropna():
    items = to_list(ing)
    all_ingredients.extend(str(i).strip().lower() for i in items if str(i).strip())

unique_ingredients = sorted(set(all_ingredients))
print("Ingredientes únicos:", len(unique_ingredients))
print(unique_ingredients[:20])

                                         name  \
0  arriba   baked winter squash mexican style   
1            a bit different  breakfast pizza   
2                   all in the kitchen  chili   
3                          alouette  potatoes   
4          amish  tomato ketchup  for canning   

                                         ingredients  \
0  ['winter squash', 'mexican seasoning', 'mixed ...   
1  ['prepared pizza crust', 'sausage patty', 'egg...   
2  ['ground beef', 'yellow onions', 'diced tomato...   
3  ['spreadable cheese with garlic and herbs', 'n...   
4  ['tomato juice', 'apple cider vinegar', 'sugar...   

                                               steps  
0  ['make a choice and proceed with recipe', 'dep...  
1  ['preheat oven to 425 degrees f', 'press dough...  
2  ['brown ground beef in large pot', 'add choppe...  
3  ['place potatoes in a large pot of lightly sal...  
4  ['mix all ingredients& boil for 2 1 / 2 hours ...  
Ingredientes únicos: 14942
['1% fat bu

In [13]:
nlp = spacy.load("en_core_web_sm")

if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner")
else:
    ner = nlp.get_pipe("ner")
ner.add_label("INGREDIENT")

TRAIN_DATA = []

# muestra de recetas (ajusta 8000 si quieres más/menos)
sample_df = df.dropna(subset=["ingredients","steps"]).sample(8000, random_state=42)

for _, row in sample_df.iterrows():
    ingredients = [str(i).strip().lower() for i in to_list(row["ingredients"]) if str(i).strip()]
    if not ingredients:
        continue

    text = steps_to_text(row["steps"])
    text_low = text.lower()
    entities = []

    for ingr in ingredients:
        start = text_low.find(ingr)
        if start != -1:
            end = start + len(ingr)
            entities.append((start, end, "INGREDIENT"))

    if not entities:
        continue

    # quitar solapes
    entities = sorted(entities, key=lambda x: (x[0], -(x[1]-x[0])))
    clean = []
    last_end = -1
    for start, end, label in entities:
        if start >= last_end:
            clean.append((start, end, label))
            last_end = end
    entities = clean
    if not entities:
        continue

    TRAIN_DATA.append((text, {"entities": entities}))

print("Ejemplos generados (antes de limpiar):", len(TRAIN_DATA))
print(TRAIN_DATA[0][0][:200], "\n", TRAIN_DATA[0][1])


Ejemplos generados (antes de limpiar): 7061
heat over to 375 degrees spray large cookie sheet with non-stick cooking spray in small bowl , combine crabmeat , cream cheese , onions and garlic salt and mix well unroll both cans of dough separate  
 {'entities': [(103, 111, 'INGREDIENT'), (114, 126, 'INGREDIENT'), (140, 151, 'INGREDIENT'), (502, 510, 'INGREDIENT'), (515, 520, 'INGREDIENT')]}


In [16]:
other_pipes = [p for p in nlp.pipe_names if p != "ner"]
with nlp.disable_pipes(*other_pipes):
    optimizer = nlp.begin_training()
    n_iter = 8  # ajusta si quieres
    for itn in range(n_iter):
        random.shuffle(TRAIN_DATA)
        losses = {}
        batches = minibatch(TRAIN_DATA, size=compounding(4.0, 32.0, 1.001))
        for batch in batches:
            examples = []
            for text, annotations in batch:
                doc = nlp.make_doc(text)
                examples.append(Example.from_dict(doc, annotations))
            nlp.update(examples, sgd=optimizer, losses=losses)
        print(f"Iteración {itn+1}/{n_iter}  Losses: {losses}")


Iteración 1/8  Losses: {'ner': 33818.82306756508}
Iteración 2/8  Losses: {'ner': 26439.565621009282}
Iteración 3/8  Losses: {'ner': 24980.13411302521}
Iteración 4/8  Losses: {'ner': 23982.090500156162}
Iteración 5/8  Losses: {'ner': 22836.621007621652}
Iteración 6/8  Losses: {'ner': 21895.73678238971}
Iteración 7/8  Losses: {'ner': 20666.84234242463}
Iteración 8/8  Losses: {'ner': 19684.77070787625}


In [17]:
output_dir = "/content/drive/MyDrive/DIET-IA/models/Adri/ner_ingredients_real_v2"
nlp.to_disk(output_dir)
print("Modelo guardado en:", output_dir)

Modelo guardado en: /content/drive/MyDrive/DIET-IA/models/Adri/ner_ingredients_real_v2


In [28]:
import spacy

output_dir = "/content/drive/MyDrive/DIET-IA/models/Adri/ner_ingredients_real_v2"
ner_ing = spacy.load(output_dir)


In [34]:
unique_ingredients_sorted = sorted(unique_ingredients, key=len, reverse=True)

def extract_ingredients_hybrid(text: str):
    text_low = text.lower()
    doc = ner_ing(text_low)
    ents = {ent.text for ent in doc.ents if ent.label_ == "INGREDIENT"}

    # diccionario
    for ingr in unique_ingredients_sorted:
        if ingr in text_low and ingr not in ents:
            ents.add(ingr)

    # limpiar subcadenas tipo "ice" dentro de "rice"
    ents = clean_substrings(list(ents))
    return ents

print(extract_ingredients_hybrid("I have tomatoes, potatoes, olive oil and peppers at home."))
print(extract_ingredients_hybrid("cook the chicken with garlic, onion and rice in a large pot."))


['olive oil', 'potatoes', 'tomatoes', 'pepper']
['chicken', 'garlic', 'onion', 'rice']


In [40]:
print(extract_ingredients_hybrid(
    "Tengo patatas, tomate y arroz." # En español no funciona :)
))

print(extract_ingredients_hybrid(
    "For breakfast I ate oats with milk, banana and a spoon of peanut butter."
))

print(extract_ingredients_hybrid(
    "We can cook a salad with lettuce, cucumber, tuna, olives and olive oil."
))

print(extract_ingredients_hybrid(
    "There is some beef in the fridge, plus onions, carrots and potatoes."
))

print(extract_ingredients_hybrid(
    "Let's bake a cake with flour, sugar, eggs, butter and chocolate chips."
))


[]
['peanut butter', 'banana', 'oats', 'milk']
['olive oil', 'cucumber', 'lettuce', 'olives', 'tuna']
['potatoes', 'carrots', 'onions', 'beef']
['chocolate chips', 'butter', 'sugar', 'flour', 'cake', 'eggs']
